In [12]:
import os
import json
from getpass import getpass

# --- Supabase & Drive Setup ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()

os.makedirs(BASE_DIR, exist_ok=True)
CREDS_FILE = os.path.join(BASE_DIR, 'supabase_creds.json')

if not os.path.exists(CREDS_FILE):
    print("Supabase credentials not found. Let's set them up.")
    print("Get these from Supabase Dashboard -> Project Settings -> API")
    url = input("Enter Supabase Project URL: ").strip()
    key = getpass("Enter Supabase SERVICE_ROLE Key (for Python backend): ").strip()
    with open(CREDS_FILE, 'w') as f:
        json.dump({'url': url, 'key': key}, f)
    print(f"\n✅ Saved credentials to {CREDS_FILE}. Your colleagues can run this and enter their own keys without changing the code!")
else:
    print(f"✅ Supabase credentials found at {CREDS_FILE}")

# Install requirements
%pip install ta supabase


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Supabase credentials found at /content/drive/MyDrive/CryptoProject/supabase_creds.json


In [ ]:
import os
import sys
import time
import json
import requests
import signal
from supabase import create_client, Client
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import ta
import math
from datetime import datetime
from sklearn.preprocessing import StandardScaler

# --- 1. Cloud & Path Configuration ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE_DIR = '/content/drive/MyDrive/CryptoProject'
    print("✅ Running in Google Colab (Drive Mounted).")
except ImportError:
    DRIVE_BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    print("✅ Running locally (No Colab detected).")

# GPU Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Utilizing Compute Device: {DEVICE}")

# File Paths
MODEL_DIR = os.path.join(DRIVE_BASE_DIR, 'models')
DATA_DIR = os.path.join(DRIVE_BASE_DIR, 'data')
CSV_PATH = os.path.join(DATA_DIR, 'BTCUSDT_5m_data.csv')

# Supabase Configuration
CREDS_FILE = os.path.join(DRIVE_BASE_DIR, 'supabase_creds.json')
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(CREDS_FILE):
    with open(CREDS_FILE, 'r') as f:
        creds = json.load(f)
    supabase: Client = create_client(creds['url'], creds['key'])
else:
    print(f"⚠️ WARNING: {CREDS_FILE} not found. Will not push to database.")
    supabase = None

# --- Constants ---
SYMBOL = 'BTCUSDT'
INTERVAL = '5m'
SEQ_LENGTH = 120
BASE_URL = "https://api.binance.com/api/v3/klines"

# --- 2. Model Architectures ---

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.1):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

class GLU(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Linear(input_size, input_size * 2)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = self.fc(x)
        content, gate = torch.chunk(x, 2, dim=-1)
        return content * self.sigmoid(gate)

class GRN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1):
        super().__init__()
        output_size = output_size or input_size
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.glu = GLU(output_size)
        self.layer_norm = nn.LayerNorm(output_size)
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Linear(input_size, output_size) if input_size != output_size else nn.Identity()
    def forward(self, x):
        residual = self.skip(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.dropout(x)
        x = self.glu(x)
        return self.layer_norm(residual + x)

class VariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim, num_vars, d_model, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_vars = num_vars
        self.grns = nn.ModuleList([GRN(input_dim // num_vars, d_model, d_model, dropout) for _ in range(num_vars)])
        self.selector_grn = GRN(input_dim, d_model, num_vars, dropout)
        self.softmax = nn.Softmax(dim=-1)
    def forward(self, x):
        weights = self.softmax(self.selector_grn(x))
        var_outputs = []
        chunk_size = x.shape[-1] // self.num_vars
        for i in range(self.num_vars):
            var_x = x[..., i*chunk_size : (i+1)*chunk_size]
            var_outputs.append(self.grns[i](var_x))
        var_outputs = torch.stack(var_outputs, dim=-1)
        selected_output = torch.sum(var_outputs * weights.unsqueeze(-2), dim=-1)
        return selected_output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1), : ]

class TFTModel(nn.Module):
    def __init__(self, input_dim, num_vars, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(input_dim, num_vars, d_model, dropout)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        x = self.vsn(x)
        x = self.pos_encoder(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])


# --- 3. Global State (In-Memory Caching) ---
df_hist = pd.DataFrame()
df_pred_log = pd.DataFrame(columns=['target_time', 'LSTM', 'Transformer'])
PRED_CSV_PATH = os.path.join(DATA_DIR, 'predictions_log.csv')

lstm_scaler = StandardScaler()
tft_scaler = StandardScaler()
model_lstm = None
model_tft = None

# LSTM expects the first 9 features
LSTM_FEATURES = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist'
]

# Transformer expects all 14 features
TFT_FEATURES = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist', 'volume_z', 'vol_spike', 'adx',
    'hour_sin', 'hour_cos'
]

# --- 4. Core Functions ---

def fetch_latest_data(symbol=SYMBOL, interval=INTERVAL, limit=1000):
    params = {'symbol': symbol, 'interval': interval, 'limit': limit}
    response = requests.get(BASE_URL, params=params)
    if response.status_code == 200:
        data = response.json()
        cols = ['open_time', 'open', 'high', 'low', 'close', 'volume',
                'close_time', 'quote_asset_volume', 'number_of_trades',
                'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore']
        df = pd.DataFrame(data, columns=cols)
        numeric_cols = ['open', 'high', 'low', 'close', 'volume']
        df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
        df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
        return df
    else:
        raise Exception(f"Failed to fetch data from Binance: {response.text}")

def calculate_features(df):
    df = df.copy()
    df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

    df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
    df['rsi_change'] = df['rsi'].diff(periods=3)
    df['rsi_accel'] = df['rsi_change'].diff(periods=2)

    macd = ta.trend.MACD(df['close'])
    macd_raw = macd.macd_diff()
    df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
    df['macd_diff'] = macd.macd_diff()
    df['macd_slope'] = df['macd_diff'].diff(periods=2)

    bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
    df['bb_pband'] = bb.bollinger_pband()
    df['bb_pband_change'] = df['bb_pband'].diff(periods=1)
    df['bb_hband'] = bb.bollinger_hband()
    df['bb_lband'] = bb.bollinger_lband()

    df['volume_raw'] = df['volume']
    df['volume'] = np.log(df['volume'] + 1)
    df['volume'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-9)

    df['vol_ma'] = df['volume'].rolling(window=20).mean()
    df['vol_std'] = df['volume'].rolling(window=20).std()
    df['volume_z'] = (df['volume'] - df['vol_ma']) / (df['vol_std'] + 1e-9)
    df['vol_spike'] = (df['volume'] > (df['vol_ma'] * 2)).astype(float)

    df['ma_20'] = df['close'].rolling(window=20).mean()
    df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9) * 10.0

    adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
    df['adx'] = adx.adx()

    df['hour'] = df['open_time'].dt.hour
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    return df.dropna()

def initialize():
    global df_hist, lstm_scaler, tft_scaler, model_lstm, model_tft

    print("\n[Init] Loading models into memory & moving to GPU...")
    lstm_path = os.path.join(MODEL_DIR, 'best_lstm_model.pth')
    if os.path.exists(lstm_path):
        model_lstm = LSTMModel(input_size=len(TFT_FEATURES))  # Changed to 14 to match checkpoint
        model_lstm.load_state_dict(torch.load(lstm_path, map_location=DEVICE))
        model_lstm.to(DEVICE)
        model_lstm.eval()
        print(f"   -> LSTM loaded on {DEVICE}.")

    tft_path = os.path.join(MODEL_DIR, 'best_tft_vsn.pth')
    if os.path.exists(tft_path):
        model_tft = TFTModel(input_dim=len(TFT_FEATURES), num_vars=len(TFT_FEATURES))
        model_tft.load_state_dict(torch.load(tft_path, map_location=DEVICE))
        model_tft.to(DEVICE)
        model_tft.eval()
        print(f"   -> Transformer loaded on {DEVICE}.")

    print(f"\n[Init] Loading historical data from Drive ({CSV_PATH})...")
    if os.path.exists(CSV_PATH):
        df_hist = pd.read_csv(CSV_PATH)
        df_hist['open_time'] = pd.to_datetime(df_hist['open_time'])
        print(f"   -> Loaded {len(df_hist)} historical rows.")

        if os.path.exists(PRED_CSV_PATH):
            global df_pred_log
            df_pred_log = pd.read_csv(PRED_CSV_PATH)
            df_pred_log['target_time'] = pd.to_datetime(df_pred_log['target_time'])
            print(f"   -> Loaded {len(df_pred_log)} past predictions.")

        print("\n[Init] Fitting scalers on training split... (this takes a moment)")
        df_full_feat = calculate_features(df_hist)
        train_end = int(len(df_full_feat) * 0.8)
        df_train = df_full_feat.iloc[:train_end]

        lstm_scaler.fit(df_train[LSTM_FEATURES])
        tft_scaler.fit(df_train[TFT_FEATURES])
        print("   -> Scalers fitted.")
    else:
        print("   -> ⚠️ No historical data found! Starting fresh.")

def save_data_to_drive():
    global df_hist
    if df_hist is not None and not df_hist.empty:
        print(f"\n🛑 [Session End] Saving updated historical data to Drive...")
        df_hist.to_csv(CSV_PATH, index=False)
        print(f"✅ Save complete! {len(df_hist)} rows written to {CSV_PATH}.")

def push_to_supabase(results):
    if not supabase:
        return
    print("   -> 🌐 Pushing updates to Supabase...")
    try:
        supabase.table('predictions').upsert({"id": 1, "payload": results}).execute()
        print("   -> ✅ Successfully pushed to Supabase!")
    except Exception as e:
        print(f"   -> ❌ Supabase push failed: {e}")

def signal_handler(sig, frame):
    save_data_to_drive()
    sys.exit(0)

# Register Graceful Shutdown
signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

# --- 5. Main Loop ---

def run_inference():
    global df_hist
    print(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Fetching latest data...")

    # 1. Fetch & Append Data
    df_new = fetch_latest_data()
    if df_hist.empty:
        df_hist = df_new
    else:
        df_hist = pd.concat([df_hist, df_new]).drop_duplicates(subset=['open_time'], keep='last').sort_values('open_time')

    # 2. Calculate Features
    df_recent = df_hist.tail(1500).copy()
    df_features = calculate_features(df_recent)

    if len(df_features) < SEQ_LENGTH:
        print("   -> ⏳ Not enough data for inference yet.")
        return

    last_price = float(df_features['close'].iloc[-1])
    results = {
        "timestamp": str(df_features['open_time'].iloc[-1]),
        "last_price": last_price,
        "predictions": {}
    }

    l_band = df_features['bb_lband'].iloc[-1]
    h_band = df_features['bb_hband'].iloc[-1]

    # 4. Infer LSTM
    if model_lstm:
        seq_lstm = df_features[TFT_FEATURES].tail(SEQ_LENGTH).values
        input_lstm = torch.tensor(tft_scaler.transform(seq_lstm), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred_log_ret = model_lstm(input_lstm).item()

        pred_price_lstm = last_price * math.exp(pred_log_ret)
        results["predictions"]["LSTM"] = {
            "val": pred_log_ret,
            "price": pred_price_lstm,
            "change_pct": (pred_price_lstm - last_price) / last_price * 100
        }

    # 5. Infer Transformer
    if model_tft:
        seq_tft = df_features[TFT_FEATURES].tail(SEQ_LENGTH).values
        # Scale and move to GPU
        input_tft = torch.tensor(tft_scaler.transform(seq_tft), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pred_bb = model_tft(input_tft).item()

        pred_price_tft = l_band + (pred_bb * (h_band - l_band))
        results["predictions"]["Transformer"] = {
            "val": pred_bb,
            "price": pred_price_tft,
            "change_pct": (pred_price_tft - last_price) / last_price * 100
        }

    # 6. Decide Consensus
    if "LSTM" in results["predictions"] and "Transformer" in results["predictions"]:
        lstm_c = abs(results["predictions"]["LSTM"]["change_pct"])
        tft_c = abs(results["predictions"]["Transformer"]["change_pct"])
        results["chosen_model"] = "LSTM" if lstm_c > tft_c else "Transformer"
    elif "LSTM" in results["predictions"]:
        results["chosen_model"] = "LSTM"
    elif "Transformer" in results["predictions"]:
        results["chosen_model"] = "Transformer"
    else:
        results["chosen_model"] = "None"

    # 7. Add History Aggregation (Sampled to keep payload small)
    try:
        # 5m: Last 500 candles
        h5 = df_hist.tail(500)[['open_time', 'close']].copy()
        h5['t'] = h5['open_time'].dt.strftime('%H:%M')

        # 1h: Last 30 days (sampled/resampled)
        df_res_1h = df_hist.set_index('open_time').resample('1h').last().dropna().reset_index()
        h1h = df_res_1h.tail(720)[['open_time', 'close']].copy()
        h1h['t'] = h1h['open_time'].dt.strftime('%m-%d %H:00')

        # 1d: Full history
        df_res_1d = df_hist.set_index('open_time').resample('1d').last().dropna().reset_index()
        h1d = df_res_1d[['open_time', 'close']].copy()
        h1d['t'] = h1d['open_time'].dt.strftime('%Y-%m-%d')

        results['history'] = {
            "5m": h5[['t', 'close']].rename(columns={'close':'p'}).to_dict('records'),
            "1h": h1h[['t', 'close']].rename(columns={'close':'p'}).to_dict('records'),
            "1d": h1d[['t', 'close']].rename(columns={'close':'p'}).to_dict('records')
        }
    except Exception as e:
        print(f"   -> History aggregation failed: {e}")

    # 8. Push directly to Supabase
    push_to_supabase(results)

    if results["chosen_model"] != "None":
        best = results['predictions'][results['chosen_model']]
        print(f"   -> AI Prediction [{results['chosen_model']}]: ${best['price']:.2f} ({best['change_pct']:+.2f}%)")

if __name__ == "__main__":
    print("========================================")
    print("   🚀 Crypto AI Prediction Cloud Engine 🚀")
    print("========================================")
    print("Press Ctrl+C to stop the engine and save data safely to Drive.")

    initialize()

    while True:
        try:
            run_inference()
        except Exception as e:
            print(f"⚠️ Error during inference: {e}")

        print("   -> Sleeping for 5 minutes...\n")
        time.sleep(300)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Running in Google Colab (Drive Mounted).
✅ Utilizing Compute Device: cuda
   🚀 Crypto AI Prediction Cloud Engine 🚀
Press Ctrl+C to stop the engine and save data safely to Drive.

[Init] Loading models into memory & moving to GPU...
   -> LSTM loaded on cuda.
   -> Transformer loaded on cuda.

[Init] Loading historical data from Drive (/content/drive/MyDrive/CryptoProject/data/BTCUSDT_5m_data.csv)...
   -> Loaded 909799 historical rows.

[Init] Fitting scalers on training split... (this takes a moment)
   -> Scalers fitted.

[2026-05-09 16:42:11] Fetching latest data...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


   -> 🌐 Pushing updates to Supabase...
   -> ✅ Successfully pushed to Supabase!
   -> AI Prediction [LSTM]: $80607.86 (-0.02%)
   -> Sleeping for 5 minutes...


[2026-05-09 16:47:15] Fetching latest data...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


   -> 🌐 Pushing updates to Supabase...
   -> ✅ Successfully pushed to Supabase!
   -> AI Prediction [LSTM]: $80646.20 (+0.01%)
   -> Sleeping for 5 minutes...


[2026-05-09 16:52:19] Fetching latest data...


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


   -> 🌐 Pushing updates to Supabase...
   -> ✅ Successfully pushed to Supabase!
   -> AI Prediction [LSTM]: $81059.43 (+0.06%)
   -> Sleeping for 5 minutes...


[2026-05-09 18:33:29] Fetching latest data...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


   -> 🌐 Pushing updates to Supabase...
   -> ✅ Successfully pushed to Supabase!
   -> AI Prediction [LSTM]: $81050.64 (+0.03%)
   -> Sleeping for 5 minutes...



: 